# Decision-tree splits: Gini, entropy, and information gain

Companion to HTB Academy Module 290, Section 6. The numbers are the course's 50-row example; we compute one **candidate** split, not a whole tree or a best-feature search.

Read $p_i$ as *p sub i*, the fraction of rows in class $i$. Read $G(S)=1-\sum_i p_i^2$ as *Gini of S equals one minus the sum over i of p sub i squared*. Read $H(S)=-\sum_i p_i\log_2p_i$ as *entropy of S equals negative the sum over i of p sub i times log base two of p sub i*. Both measure class mixing.

In [ ]:
from math import log2

def proportions(class_counts):
    # Each count corresponds to one class. Zero-count classes are harmless.
    total = sum(class_counts)
    assert total > 0, 'a node must contain at least one row'
    return [count / total for count in class_counts]

def gini(class_counts):
    # Formula: G(S) = 1 - sum_i p_i^2.
    return 1.0 - sum(p ** 2 for p in proportions(class_counts))

def entropy(class_counts):
    # Formula: H(S) = -sum_i p_i*log2(p_i).
    # Skip p=0: its mathematical limiting contribution is zero, but log2(0) is undefined.
    return -sum(p * log2(p) for p in proportions(class_counts) if p > 0)

parent = (30, 20)  # Class A, class B.
print('parent Gini:', gini(parent))
print('parent entropy:', entropy(parent))
assert abs(gini(parent) - 0.48) < 1e-12
assert abs(entropy((5, 0))) < 1e-12  # A pure node has zero uncertainty.

## Score HTB's proposed feature split

Read $IG(S,F)=H(S)-\sum_v (|S_v|/|S|)H(S_v)$ as *information gain of S from F equals parent entropy minus the sum over branches v of the child's size fraction times its entropy*. The weighting matters: the 30-row branch must count more than the 20-row branch. A high gain means a large reduction in class uncertainty.

In [ ]:
# F=1 has 20 A and 10 B; F=2 has 10 A and 10 B.
children = [(20, 10), (10, 10)]
assert tuple(sum(child[i] for child in children) for i in range(2)) == parent
parent_size = sum(parent)

# Formula: weighted child entropy = sum_v (|S_v|/|S|)*H(S_v).
weighted_entropy = sum((sum(child) / parent_size) * entropy(child) for child in children)
information_gain = entropy(parent) - weighted_entropy
print('branch entropies:', [round(entropy(child), 6) for child in children])
print('weighted child entropy:', round(weighted_entropy, 6))
print('information gain:', round(information_gain, 6))
assert abs(entropy(parent) - 0.9709505944546686) < 1e-12
assert abs(weighted_entropy - 0.9509775004326937) < 1e-12
assert abs(information_gain - 0.0199730940219749) < 1e-12

This split has a small positive gain. We have **not** compared it with any other feature, grown a tree, or measured new-data accuracy. In a real security workflow, split quality on training records is only one part of evaluation; data leakage, poisoning, and overfitting remain possible.